# KAVACH v3: GPU training + interpretability suite (Colab Pro)

**Runtime:** *Runtime > Change runtime type > GPU (A100 or L4 if offered) + High-RAM.*
**Before running:** upload `kavach_backend.zip` to `MyDrive/kavach/`.

## What changed vs. the repo's original pipeline (and why)
| Change | Why | Expected benefit |
|---|---|---|
| **GPU training** (`--device auto`): model, heads, losses and *all cached graphs* live on the GPU | the HGT/GRU math and PyG batching were CPU-only; graphs were re-collated on CPU every step | faster epochs, so more epochs / bigger batches fit in the same time. Checkpoints are saved as CPU tensors so the API still loads them |
| **Input-noise augmentation** (`--noise-std 0.1`, context encoder only) | `ENHANCEMENTS.md` reports PGD flips detections with ~0.06 sigma perturbations | a flatter decision surface; verified by re-running the adversarial benchmark (cell 12). A measurement, not a promise |
| **Bigger batch (128), 20 epochs** | more stable VICReg variance/covariance statistics per step | less collapse risk; compare the calibration F1 curve in the log |
| **Drive checkpoints after each stage** | Colab VMs vanish | resume without retraining |
| **Interpretability visuals** (probe heatmap, SAE labels) | the suite only produced JSON | see *where* concepts become linearly readable and what SAE features fire on |

Architecture, losses, heads, SAE, lens, probes and concept bottleneck are unchanged. No performance claim is made until you read the printed metrics.

In [ ]:
import torch, os
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| cpus', os.cpu_count())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
else: print('WARNING: no GPU - change runtime type. Training still works on CPU.')
torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True

In [ ]:
# 1. Drive + code
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/kavach'
os.makedirs(DRIVE + '/outputs', exist_ok=True)
!rm -rf /content/work && mkdir -p /content/work && unzip -q {DRIVE}/kavach_backend.zip -d /content/work
%cd /content/work/backend

In [ ]:
# 2. Dependencies (recent torch-geometric needs no custom wheels for HGTConv)
!pip -q install torch-geometric captum shap scikit-learn pandas pyyaml scapy sqlalchemy fastapi pytest matplotlib
import torch_geometric; print('pyg', torch_geometric.__version__)

In [ ]:
# 3. Data: CIC-IDS-2017 MachineLearningCVE (8 CSVs). Tries, in order: (a) your Drive cache, (b) the UNB server.
import glob, shutil, zipfile
CSV_DIR = 'data/raw/CSV/MachineLearningCVE'; os.makedirs(CSV_DIR, exist_ok=True)
CACHE = DRIVE + '/data_csv'          # put the 8 CSVs here once and every future run reuses them
EXPECTED = ['Monday-WorkingHours', 'Tuesday-WorkingHours', 'Wednesday-workingHours', 'Thursday-WorkingHours-Morning-WebAttacks',
            'Thursday-WorkingHours-Afternoon-Infilteration', 'Friday-WorkingHours-Morning', 'Friday-WorkingHours-Afternoon-PortScan', 'Friday-WorkingHours-Afternoon-DDos']
def have_data():
    fs = glob.glob(CSV_DIR + '/*.csv')
    return len(fs) == 8 and all(os.path.getsize(f) > 1e6 for f in fs)
if glob.glob(CACHE + '/*.csv'):
    for f in glob.glob(CACHE + '/*.csv'): shutil.copy(f, CSV_DIR)
    print('copied from Drive cache')
if not have_data():
    !wget -q -T 60 -O /content/mlcve.zip http://205.174.165.80/CICDataset/CIC-IDS-2017/Dataset/MachineLearningCVE.zip
    if zipfile.is_zipfile('/content/mlcve.zip'):
        zipfile.ZipFile('/content/mlcve.zip').extractall('/content/mlcve')
        for f in glob.glob('/content/mlcve/**/*.csv', recursive=True): shutil.move(f, CSV_DIR)
    else:
        print('UNB download is not a zip (size %d bytes) - use cell 3b (Kaggle) or put the CSVs in %s' % (os.path.getsize('/content/mlcve.zip'), CACHE))
files = sorted(glob.glob(CSV_DIR + '/*.csv'))
for f in files: print(f'{os.path.getsize(f)/1e6:8.1f} MB  {os.path.basename(f)}')
print('data ready' if have_data() else 'DATA MISSING - run cell 3b, then re-run this cell')

In [ ]:
# 3b. Kaggle fallback (skips itself if data is already there). One-time setup:
#   kaggle.com > Settings > API > 'Create New Token' downloads kaggle.json; drag it into Colab's file panel (/content).
if not have_data():
    !mkdir -p ~/.kaggle && cp /content/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip -q install kaggle
    !kaggle datasets download -d chethuhn/network-intrusion-dataset -p /content/kg --unzip
    for f in glob.glob('/content/kg/**/*.csv', recursive=True): shutil.copy(f, CSV_DIR)
    os.makedirs(CACHE, exist_ok=True)
    for f in glob.glob(CSV_DIR + '/*.csv'): shutil.copy(f, CACHE)   # cache on Drive for next time
have = {os.path.basename(f).split('.pcap')[0] for f in glob.glob(CSV_DIR + '/*.csv')}
print('missing expected files:', [e for e in EXPECTED if e not in have] or 'none')
assert have_data(), 'still no data - see the instructions printed above'

In [ ]:
# 4. Unit tests (catch env problems before a long run)
!python -m pytest -q -x tests 2>&1 | tail -15

In [ ]:
# 5. Build windows (CPU-bound, a few minutes)
!python -m src.training.build_windows_cache

In [ ]:
# 6. GPU smoke test: 1 epoch on a small slice. Confirms device=cuda and no device-mismatch errors before the long run.
!python -m src.training.train_v2 --epochs 1 --max-train 2000 --sources csv --noise-std 0.1 --out data/processed/smoke.pt 2>&1 | tail -25

In [ ]:
# 7. Full training on GPU. skips the forensic-ledger write (remove it to chain the run).
!python -m src.training.train_v2 --epochs 20 --batch-size 128 --noise-std 0.1 --sources csv --report data/processed/train_report.json 2>&1 | tail -80

In [ ]:
# 8. Save checkpoint immediately
!rm -rf {DRIVE}/outputs/processed_after_train && cp -r data/processed {DRIVE}/outputs/processed_after_train && ls -lh data/processed | head

In [ ]:
# 9. Interpretability suite (SAE, tuned lens, probes, concept bottleneck, benign reference)
!python -m src.training.train_interpretability 2>&1 | tail -60

In [ ]:
# 10. Probe heatmap: at which layer does each concept become linearly readable? (0.5 = chance for binary)
import json, numpy as np, matplotlib.pyplot as plt
d = 'data/processed/interpretability'
pr = json.load(open(d + '/probes.json'))
layers = pr['layers']; rows = pr['concepts']
M = np.array([[r['layers'].get(l, {}).get('balanced_accuracy', np.nan) for l in layers] for r in rows])
fig, ax = plt.subplots(figsize=(1.2*len(layers)+3, 0.5*len(rows)+2))
im = ax.imshow(M, vmin=0.5, vmax=1.0, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(layers))); ax.set_xticklabels(layers, rotation=30, ha='right')
ax.set_yticks(range(len(rows))); ax.set_yticklabels([r['concept'] for r in rows])
for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        if not np.isnan(M[i, j]): ax.text(j, i, f'{M[i,j]:.2f}', ha='center', va='center', color='w', fontsize=8)
plt.colorbar(im, label='balanced accuracy'); plt.title('Linear probes: concept x layer'); plt.tight_layout(); plt.show()
for r in rows: print(f"{r['concept']:<28} best layer: {r['best_layer']}")

In [ ]:
# 11. Report + SAE feature labels
rep = json.load(open(d + '/interpretability_report.json'))
print(json.dumps(rep, indent=1)[:5000])
for f in glob.glob(d + '/*label*.json'):
    print('\n', f); print(open(f).read()[:3000])

In [ ]:
# 12. Robustness: how small a perturbation flips a detection? (compare against the ~0.06-0.07 sigma reported before noise augmentation)
!python -m src.benchmark.robustness --n 30 2>&1 | tail -40

In [ ]:
# 13. Persist everything
!rm -rf {DRIVE}/outputs/processed && cp -r data/processed {DRIVE}/outputs/processed
print('saved ->', DRIVE + '/outputs/processed')

## Optional next experiments
* **Ablate the noise:** rerun cell 7 with `--noise-std 0 --out data/processed/netjepa_nonoise.pt --report data/processed/nonoise_report.json` and compare F1 vs. PGD robustness.
* **Packet level:** download the 2 PCAPs (links in `data/raw/PCAP/README.md`), run `python -m src.training.build_pcap_windows <pcap>` for each, retrain with `--sources csv,pcap`, then rerun cell 9.
* **Generalisation:** `python -m src.benchmark.generalization --epochs 4` (leave-one-attack-out retraining).